# Montra Hospitality Intel: Topic Modelling with BERTopic
**Guest Experience Analytics & Topic Modelling**

This notebook applies BERTopic to discover latent, operational hospitality themes from unstructured guest reviews.
Geographic entity stripping is performed (removing UK city names so topics cluster around service quality rather than location), documents are embedded using Sentence Transformers (`all-MiniLM-L6-v2`), clustered via UMAP + HDBSCAN, topic representations extracted using class-based TF-IDF (c-TF-IDF), topics reduced, and outlier feedback reassigned.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earlchirchir/Guest-Experience-Analytics/blob/main/notebook/03_topic_modelling.ipynb)

> **Note**: Run this notebook on Google Colab with GPU hardware acceleration (Runtime > Change runtime type > T4 GPU) for fast transformer embedding generation and BERTopic clustering.

In [1]:
# Install dependencies in Google Colab
!pip install -q bertopic sentence-transformers umap-learn hdbscan


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 2.8 MB/s eta 0:00:00


In [2]:
import os
import re
from collections import Counter
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

# Load preprocessed guest feedback data (local workspace or Colab)
input_path = "../data/processed/montra_cleaned_data.csv"
if not os.path.exists(input_path):
    if os.path.exists("data/processed/montra_cleaned_data.csv"):
        input_path = "data/processed/montra_cleaned_data.csv"
    elif os.path.exists("montra_cleaned_data.csv"):
        input_path = "montra_cleaned_data.csv"
    else:
        input_path = "https://raw.githubusercontent.com/earlchirchir/Guest-Experience-Analytics/main/data/processed/montra_cleaned_data.csv"

merged = pd.read_csv(input_path)
print("Loaded preprocessed data shape:", merged.shape)
merged.head()


Loaded preprocessed data shape: (35000, 23)


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms,cleaned_review_text
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118,the overall price was fair for what we paid mi...
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95,the room was nothing out of the ordinary for w...
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,2,Business,NaN,Ireland,4,The London Boutique,London,4,231,unfortunately the pool was outdated it clearly...
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,Family Room,Airbnb,...,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101,we were disappointed that the neighborhood was...
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214,the room service was standard hotel fare for w...


In [3]:
# Extract review document list
document = merged['cleaned_review_text'].astype(str).tolist()
print(f"Total review documents: {len(document)}")
print("Sample documents:")
for i, doc in enumerate(document[:3]):
    print(f"[{i+1}] {doc}")


Total review documents: 35000
Sample documents:
[1] the overall price was fair for what we paid might consider it again might not
[2] the room was nothing out of the ordinary for what we paid the overall price exceeded our expectations a great deal and the quality easily matched the price
[3] unfortunately the pool was outdated it clearly needed some upkeep wouldn t recommend it to others


### 1. Remove Geographic Noise (UK Cities)
To prevent the topic model from clustering reviews by geographic location (e.g., separating London hotels from Edinburgh hotels), all UK city names are stripped out so the clusters capture operational hospitality themes.

In [4]:
# UK Cities list to eliminate geographic bias
UK_Cities = [
    "London", "Manchester", "Edinburgh", "Birmingham", "Leeds",
    "Bristol", "Liverpool", "Glasgow", "Oxford", "Bath",
    "York", "Cardiff", "New Castle", "Nottingham", "Brighton",
    "Sheffield", "Leicester", "Belfast", "Southampton", "Aberdeen"
]

# Compile case-insensitive regex pattern and remove city references
city_pattern = re.compile(r'\b(' + '|'.join(UK_Cities) + r')\b', re.IGNORECASE)
document = [city_pattern.sub('', doc).strip() for doc in document]
document = [re.sub(r'\s+', ' ', doc) for doc in document]

print("Sanitized documents preview:")
for i, doc in enumerate(document[:3]):
    print(f"[{i+1}] {doc}")


Sanitized documents preview:
[1] the overall price was fair for what we paid might consider it again might not
[2] the room was nothing out of the ordinary for what we paid the overall price exceeded our expectations a great deal and the quality easily matched the price
[3] unfortunately the pool was outdated it clearly needed some upkeep wouldn t recommend it to others


### 2. Vocabulary & Duplicate Review Inspection
Document collection vocabulary analysis and inspection of duplicated normalized reviews.

In [5]:
# Count exact duplicate normalized reviews
normalized_docs = [doc.lower().strip() for doc in document]
total_docs = len(normalized_docs)
unique_docs = len(set(normalized_docs))
duplicated_docs = total_docs - unique_docs

print(f"Total documents: {total_docs}")
print(f"Unique documents: {unique_docs}")
print(f"Duplicate documents: {duplicated_docs}")

# Check vocabulary size and top 20 most frequent words
all_words = " ".join(document).split()
print("Vocab size:", len(set(all_words)))
print("\nTop 20 most common tokens:")
print(Counter(all_words).most_common(20))


Total documents: 35000
Unique documents: 34400
Duplicate documents: 600
Vocab size: 507

Top 20 most common tokens:
[('the', 73552), ('was', 54683), ('we', 32665), ('nothing', 24495), ('our', 23913), ('in', 23668), ('to', 18777), ('for', 18311), ('what', 17937), ('a', 17697), ('and', 17560), ('that', 16124), ('about', 12575), ('overall', 12124), ('it', 11539), ('paid', 11056), ('stay', 11027), ('as', 11012), ('out', 10561), ('i', 10224)]


### 3. Generate Semantic Sentence Embeddings
The pretrained Sentence Transformer model `all-MiniLM-L6-v2` converts each review into a dense 384-dimensional semantic embedding vector.

In [6]:
from sentence_transformers import SentenceTransformer

# Load sentence transformer model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings (runs with progress bar)
embeddings = embedding_model.encode(document, show_progress_bar=True)
print("Embeddings array shape:", embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1094 [00:00<?, ?it/s]

Embeddings array shape: (35000, 384)


### 4. Build and Train BERTopic Model
Modular BERTopic architecture configuration:
1. **UMAP**: Reduces dense 384-d vectors into 5 dimensions for clustering while preserving local/global manifold structure.
2. **HDBSCAN**: Identifies dense clusters of variable shape without forcing noise into clusters.
3. **CountVectorizer**: Generates candidate n-grams (1 to 3 words) ignoring standard English stopwords.
4. **c-TF-IDF**: Scores how distinctive words are for each cluster relative to the overall corpus.

In [7]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

# 1. Dimensionality reduction with UMAP
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# 2. Density-based clustering with HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# 3. Tokenizer & Stopwords vectorizer
vectorizer_model = CountVectorizer(
    min_df=5,
    ngram_range=(1, 3),
    stop_words="english"
)

# 4. Class-based TF-IDF
ctfidf_model = ClassTfidfTransformer()

# 5. Assemble BERTopic pipeline
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=True,
    verbose=True
)

# Fit and transform
topics, probabilities = topic_model.fit_transform(document, embeddings)
print("Initial topic extraction completed!")


2026-10-20 17:59:27,691 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-20 18:00:49,115 - BERTopic - Dimensionality - Completed ✓
2026-10-20 18:00:49,122 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-20 18:04:42,988 - BERTopic - Cluster - Completed ✓
2026-10-20 18:04:43,016 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-20 18:04:44,696 - BERTopic - Representation - Completed ✓


Initial topic extraction completed!


In [8]:
# Display extracted topics overview
topic_info = topic_model.get_topic_info()
print(f"Total initial topics discovered: {len(topic_info)}")
display(topic_info.head(15))


Total initial topics discovered: 459


,Topic,Count,Name,Representation,Representative_Docs
0,-1,10184,-1_genuinely_honestly_room_paid,"[genuinely, honestly, room, paid, disappointed...",[i d describe the overall price as in line wit...
1,0,716,0_overall dcor_dcor_overall overall dcor_overa...,"[overall dcor, dcor, overall overall dcor, ove...",[nothing about our visit to stood out much eit...
2,1,415,1_housekeeping staff_housekeeping_overall hous...,"[housekeeping staff, housekeeping, overall hou...",[our stay in was fairly ordinary meanwhile ove...
3,2,399,2_night_night staff_staff_night staff fine,"[night, night staff, staff, night staff fine, ...",[nothing about our visit to stood out much eit...
4,3,276,3_business center_center_business_overall busi...,"[business center, center, business, overall bu...",[nothing about our visit to stood out much eit...
5,4,258,4_bathroom_bathroom area_overall bathroom_stoo...,"[bathroom, bathroom area, overall bathroom, st...",[nothing about our visit to stood out much eit...
6,5,253,5_pool_pool basic_pool basic functional_overal...,"[pool, pool basic, pool basic functional, over...",[nothing about our visit to stood out much eit...
7,6,233,6_overall getting hotel_hotel staff_getting ho...,"[overall getting hotel, hotel staff, getting h...",[overall getting around from the hotel was not...
8,7,211,7_spa_spa left lot_spa left_disappointed spa,"[spa, spa left lot, spa left, disappointed spa...",[our time in got off to a rocky start here tha...
9,8,207,8_buffet_overall buffet_buffet standard_buffet...,"[buffet, overall buffet, buffet standard, buff...",[overall the buffet was standard hotel fare ab...


### 5. Topic Reduction & Outlier Reassignment
Topic reduction and outlier reassignment:
1. Redundant micro-topics are reduced using `reduce_topics(document, nr_topics='auto')` or by setting a target number of operational hospitality clusters (e.g., 7 major pillars: Staff, Cleanliness, Comfort, Dining, Amenities, Location, Value).
2. Outlier reviews (topic -1) are reassigned to their closest semantic topic based on probabilities.

In [9]:
# Reduce topics automatically to consolidate granular sub-themes
topic_model.reduce_topics(document, nr_topics="auto")

# Reassign outlier documents (topic -1) to their highest probability topic
new_topics = topic_model.reduce_outliers(document, topics, strategy="c-tf-idf")
topic_model.update_topics(document, topics=new_topics)

print("Updated Topic Information after reduction and outlier handling:")
topic_info_reduced = topic_model.get_topic_info()
display(topic_info_reduced.head(10))


2026-10-20 18:04:47,546 - BERTopic - Topic reduction - Reducing number of topics
2026-10-20 18:04:48,031 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-20 18:04:49,626 - BERTopic - Representation - Completed ✓
2026-10-20 18:04:49,636 - BERTopic - Topic reduction - Reduced number of topics from 459 to 20
2026-10-20 18:04:51,058 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Updated Topic Information after reduction and outlier handling:


,Topic,Count,Name,Representation,Representative_Docs
0,0,5989,0_dcor_clean_overall_what,"[dcor, clean, overall, what, nothing, room, ab...",[this hotel was a fine if unremarkable choice ...
1,1,587,1_housekeeping_dcor_staff_overall,"[housekeeping, dcor, staff, overall, professio...",[the pricing was in line with similar hotels n...
2,2,535,2_night_hard_staff_reach,"[night, hard, staff, reach, car, without, comp...",[getting around from the hotel was hard to rea...
3,3,306,3_center_business_basic_ages,"[center, business, basic, ages, city, needs, o...",[the location left a lot to be desired it took...
4,4,480,4_bathroom_taxi_take_everywhere,"[bathroom, taxi, take, everywhere, desired, lo...",[the neighborhood left a lot to be desired we ...
5,5,912,5_bargain_rip_like_or,"[bargain, rip, like, or, felt, pool, manager, ...",[the manager was fine nothing special for what...
6,6,321,6_dark_little_around_after,"[dark, little, around, after, from, very, gett...",[the neighborhood left a lot to be desired the...
7,7,282,7_spa_40_waited_over,"[spa, 40, waited, over, minutes, serious, refu...",[our arrival experience left a lot to be desir...
8,8,495,8_buffet_team_pretty_feeling,"[buffet, team, pretty, feeling, service, disap...",[this wasn t the experience we expected during...
9,9,255,9_carpet_system_apparently_crashed,"[carpet, system, apparently, crashed, waited, ...",[our time in got off to a rocky start here tha...


### 6. Topic Visualizations
Visual diagnostics to explore topic semantics and intertopic relationships:
- Topic word score barcharts
- Intertopic distance maps
- Topic hierarchy dendrogram

In [10]:
# Top terms per topic barchart
fig_barchart = topic_model.visualize_barchart(top_n_topics=8, n_words=8)
fig_barchart.show()


In [15]:
import numpy as np

# Recalculate topic embeddings manually to align with the 20 newly reduced topics
unique_topics = sorted(list(set(new_topics)))
topic_embeddings = []
for topic in unique_topics:
    # Get embeddings belonging to this topic
    indices = [i for i, t in enumerate(new_topics) if t == topic]
    topic_centroid = np.mean(embeddings[indices], axis=0)
    topic_embeddings.append(topic_centroid)

topic_model.topic_embeddings_ = np.array(topic_embeddings)

# Generate and show Intertopic distance map
fig_topics = topic_model.visualize_topics()
fig_topics.show()

In [16]:
# Topic hierarchy dendrogram
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy.show()


### 7. Export Topic Modelling Results
Assigned topic IDs and representative keywords are attached back to the original dataset and saved to `../data/processed/reviews_with_topics.csv`.

In [18]:
# Map topic labels
topic_labels = {
    row['Topic']: row['Name'] for _, row in topic_model.get_topic_info().iterrows()
}

merged['topic_id'] = new_topics
merged['topic_name'] = merged['topic_id'].map(topic_labels)

# Save enriched dataset
output_path = "../data/processed/reviews_with_topics.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged.to_csv(output_path, index=False)
print(f"Successfully saved {len(merged)} reviews with topic assignments to {output_path}")

# Save trained topic model
os.makedirs("../model", exist_ok=True)
topic_model.save("../model/bertopic_model", serialization="safetensors", save_embedding_model=False)
print("Successfully saved trained topic model to ../model/bertopic_model")

# Display topic distribution
print("\nReview counts per topic:")
print(merged['topic_name'].value_counts())

2026-10-20 18:09:16,329 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Successfully saved 35000 reviews with topic assignments to ../data/processed/reviews_with_topics.csv
Successfully saved trained topic model to ../model/bertopic_model

Review counts per topic:
topic_name
0_dcor_clean_overall_what                   5989
16_quality_given_affordable_easily          1069
5_bargain_rip_like_or                        912
18_city_ages_took_checked                    609
1_housekeeping_dcor_staff_overall            587
                                            ... 
454_cafe_house_check_process                  15
453_buffet_overpriced_bland_affected          15
455_neighborhood_bathroom_clean_passably      15
456_pool_needs_check_desk                     15
457_luggage_help_their_went                   15
Name: count, Length: 458, dtype: int64
